## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Nota importante - Cargo API de WebSearchTool

Esto me está costando 2,5 céntimos por llamada para OpenAI WebSearchTool. Eso puede sumar entre 2 y 3 dólares para los próximos 2 laboratorios. Utilizaremos herramientas de búsqueda gratuitas y de bajo coste con otras plataformas, así que siéntete libre de no ejecutar esto si el coste es una preocupación. También el estudiante Christian W. señaló que OpenAI a veces puede cobrar por múltiples búsquedas para una sola llamada, por lo que a veces podría costar más de 2,5 centavos por llamada.



Costs are here: https://platform.openai.com/docs/pricing#web-search

In [3]:
INSTRUCTIONS = "Usted es asistente de investigación. Dado un término de búsqueda, busca ese término en Internet y \
un resumen conciso de los resultados. El resumen debe tener entre 2 y 3 párrafos y menos de 300 \
palabras. Capta los puntos principales. Escriba de forma sucinta, no es necesario que tenga frases completas o una buena \
gramática. Esto lo leerá una persona que sintetiza un informe, así que es vital que captes la esencia y dejes de lado cualquier palabrería.\
esencia e ignorar cualquier palabrería. No incluyas más comentarios que el propio resumen."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [4]:
message = "Últimos marcos para agentes de IA en 2025"

with trace("Busqueda"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

En 2025, se han desarrollado diversos marcos y herramientas para la creación y gestión de agentes de inteligencia artificial (IA). Entre ellos, Manus, lanzado el 6 de marzo de 2025 por la startup china Monica, destaca como un agente autónomo capaz de realizar tareas complejas en línea sin intervención humana directa. ([es.wikipedia.org](https://es.wikipedia.org/wiki/Manus_%28agente_de_IA%29?utm_source=openai)) Además, FlowiseAI ofrece una plataforma de código abierto y bajo código que permite a los desarrolladores diseñar flujos de trabajo de agentes de IA personalizados mediante una interfaz visual, facilitando su adopción incluso para quienes no son programadores. ([kwfoundation.org](https://kwfoundation.org/blog/2025/02/19/el-agente-de-inteligencia-artificial-significara-el-fin-del-saas/?utm_source=openai))

En el ámbito académico, se han propuesto marcos para mejorar la seguridad y la responsabilidad en el uso de agentes de IA. El artículo "Authenticated Delegation and Authorized AI Agents" introduce un sistema que permite a los usuarios delegar tareas a agentes de IA de manera segura, manteniendo una cadena clara de responsabilidad. ([arxiv.org](https://arxiv.org/abs/2501.09674?utm_source=openai)) Por su parte, "Infrastructure for AI Agents" aborda la necesidad de infraestructuras técnicas y protocolos compartidos que medien las interacciones de los agentes con su entorno, garantizando su alineación con las instituciones existentes y la detección de acciones perjudiciales. ([arxiv.org](https://arxiv.org/abs/2501.10114?utm_source=openai))

Estas iniciativas reflejan un esfuerzo global por establecer marcos y herramientas que faciliten la creación, gestión y regulación de agentes de IA, buscando equilibrar la innovación tecnológica con la seguridad y la responsabilidad. 

### As always, take a look at the trace

https://platform.openai.com/traces

### Ahora utilizaremos Structured Outputs, e incluiremos una descripción de los campos

In [5]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"Eres un útil asistente de investigación. Dada una consulta, propón una serie de búsquedas en Internet \
para responder mejor a la consulta. Salida de {HOW_MANY_SEARCHES} términos a consultar."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Su razonamiento de por qué esta búsqueda es importante para la consulta.") # Se obliga al model a pensar antes de realizar la accion

    query: str = Field(description="El término de búsqueda que se utilizará para la búsqueda web.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="Una lista de búsquedas en Internet para responder mejor a la consulta.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [6]:

message = "Últimos frameworks para agentes de IA en 2025"

with trace("Busqueda Kevin"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='Para identificar las tendencias y novedades en frameworks de IA en 2025 y cómo están evolucionando.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='Para obtener un análisis de los frameworks más populares y utilizados por desarrolladores en 2025.', query='most popular AI agent frameworks 2025'), WebSearchItem(reason='Para examinar las características y ventajas de los nuevos frameworks de IA y su impacto en el desarrollo.', query='features of new AI agent frameworks 2025')]


In [7]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Enviar un correo electrónico con el asunto y el cuerpo HTML dados """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("dnaranjo88@gmail.com") # Change this to your verified email
    to_email = To("zews.kevina@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [8]:
send_email

FunctionTool(name='send_email', description='Enviar un correo electrónico con el asunto y el cuerpo HTML dados', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E1CD926FC0>, strict_json_schema=True, is_enabled=True)

In [9]:
INSTRUCTIONS = """Podrá enviar un correo electrónico en formato HTML basado en un informe detallado.
Se le proporcionará un informe detallado. Debe utilizar su herramienta para enviar un correo electrónico, proporcionando el 
informe convertido en HTML limpio y bien presentado con una línea de asunto apropiada."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [10]:
INSTRUCTIONS = (
    "Usted es un investigador senior encargado de redactar un informe cohesionado para una consulta de investigación. "
    "Se le proporcionará la consulta original y una investigación inicial realizada por un asistente de investigación.\n"
    "En primer lugar, debe elaborar un esquema para el informe que describa la estructura y el "
    "flujo del informe. A continuación, genere el informe y envíelo como resultado final.\n"
    "El resultado final debe estar en formato markdown, y debe ser extenso y detallado. El objetivo "
    "5-10 páginas de contenido, al menos 1000 palabras."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="Un breve resumen de 2-3 frases de las conclusiones.")

    markdown_report: str = Field(description="El informe final")

    follow_up_questions: list[str] = Field(description="Temas sugeridos para seguir investigando")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### Las siguientes 3 funciones planificarán y ejecutarán la búsqueda, usando planner_agent y search_agent

In [11]:
async def plan_searches(query: str):
    """ Utilizar el agente_planificador para planificar las búsquedas que se realizarán para la consulta """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Realizará {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Llamar a search() para cada elemento del plan de búsqueda """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Búsqueda finalizada")
    return results

async def search(item: WebSearchItem):
    """ Utilice el agente de búsqueda para realizar una búsqueda web de cada elemento del plan de búsqueda """
    input = f"Término de búsqueda: {item.query}\nMotivo de la búsqueda: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [12]:
async def write_report(query: str, search_results: list[str]):
    """ Utilice el agente redactor para escribir un informe basado en los resultados de la búsqueda"""
    print("Pensando en el informe...")
    input = f"Original query: {query}\nResultados resumidos de la búsqueda: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finalizada la redacción del informe")
    return result.final_output

async def send_email(report: ReportData):
    """ Utilice el agente de correo electrónico para enviar un correo electrónico con el informe """
    print("Escribir correo electrónico...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Correo electrónico enviado")
    return report

### Showtime!

In [13]:
query ="Últimos frameworks para agentes de IA en 2025"

with trace("Rastro de investigación - Kevin"):
    print("Empezar a investigar...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hurra!")




Empezar a investigar...
Planning searches...
Realizará 3 searches
Searching...
Búsqueda finalizada
Pensando en el informe...
Finalizada la redacción del informe
Escribir correo electrónico...
Correo electrónico enviado
Hurra!


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Enhorabuena por tus progresos, y una petición</h2>
            <span style="color:#00cc00;">Ha llegado a un momento importante con el curso; ha creado un Agente valioso utilizando uno de los últimos marcos de Agente. Ha mejorado sus competencias y ha abierto nuevas posibilidades comerciales. Tómese un momento para celebrar su éxito.<br/><br/>Algo que debería pedirte -- mi editor me abofetearía si no mencionara esto. Si puedes valorar el curso en Udemy, te estaría muy agradecido: es la forma más importante en que Udemy decide si mostrar el curso a otros y supone una gran diferencia.<br/><br/>Y otro recordatorio para <a href="https://www.linkedin.com/in/eddonner/">Conéctate conmigo en LinkedIn</a> ¡si lo deseas! Si desea publicar acerca de su progreso en el curso, por favor etiquéteme y voy a sopesar para aumentar su exposición.
            </span>
        </td>
    </tr>